# Section B: RAG Fundamentals, Implement It

B1. Chunking Implement fixed-size chunking on your corpus with a chunk size of your choice and some overlap between
chunks. Print out the resulting chunks and briefly explain your choice of chunk size and overlap.


In [5]:
!pip install -q langchain-text-splitters

In [13]:
# Sample Text for Demonstration
text = """Machine learning is a branch of artificial intelligence that allows computers to learn from data.
It is used for prediction, classification, recommendation systems, and pattern recognition.

Natural language processing is a field of artificial intelligence that helps computers understand human language.
It is commonly used in chatbots, translation, sentiment analysis, and text summarization.

A vector database stores data as numerical vectors called embeddings.
It helps applications find information that is similar in meaning instead of only matching exact words.

Retrieval-Augmented Generation, or RAG, combines information retrieval with a language model.
It retrieves relevant information from a knowledge base and provides it to the model as additional context.

Chunking is an important step in a RAG pipeline because large documents are divided into smaller pieces.
Smaller chunks make it easier for the retrieval system to find the most relevant information.

Chunk overlap means that some part of one chunk is repeated in the next chunk.
This helps preserve context when important information is located near the boundary of two chunks.
"""

In [14]:
from langchain_text_splitters import CharacterTextSplitter

# Create a CharacterTextSplitter for fixed-size chunking with overlap
fixed_overlap_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=150,    # Number of characters per chunk
    chunk_overlap=50   # Number of overlapping characters between chunks
)

# Split the text into overlapping chunks
fixed_overlap_chunks = fixed_overlap_splitter.split_text(text)


In [15]:
# Display the chunks
print("Total Chunks:", len(fixed_overlap_chunks))

for i, chunk in enumerate(fixed_overlap_chunks, 1):
    print("\n" + "=" * 60)
    print(f"Chunk {i}")
    print("=" * 60)
    print(chunk)
    print("Character count:", len(chunk))

Total Chunks: 12

Chunk 1
Machine learning is a branch of artificial intelligence that allows computers to learn from data.
It is used for prediction, classification,
Character count: 140

Chunk 2
data.
It is used for prediction, classification, recommendation systems, and pattern recognition.

Natural language processing is a field of
Character count: 140

Chunk 3
language processing is a field of artificial intelligence that helps computers understand human language.
It is commonly used in chatbots,
Character count: 138

Chunk 4
human language.
It is commonly used in chatbots, translation, sentiment analysis, and text summarization.

A vector database stores data as numerical
Character count: 149

Chunk 5
vector database stores data as numerical vectors called embeddings.
It helps applications find information that is similar in meaning instead of only
Character count: 149

Chunk 6
that is similar in meaning instead of only matching exact words.

Retrieval-Augmented Generation, or RA

In [17]:
print("""
I selected a chunk size of 150 characters so that the text is divided
into small and manageable pieces. I used 50 characters of overlap so
that some context from the previous chunk is retained in the next chunk.
This helps reduce the chance of losing important information at chunk
boundaries.
""")


I selected a chunk size of 150 characters so that the text is divided
into small and manageable pieces. I used 50 characters of overlap so
that some context from the previous chunk is retained in the next chunk.
This helps reduce the chance of losing important information at chunk
boundaries.



Refrence from this link for first question:
https://medium.com/@jagadeesan.ganesh/understanding-chunking-algorithms-and-overlapping-techniques-in-natural-language-processing-df7b2c7183b2

B2. Dense retrieval Embed your chunks using any embedding model (sentence-transformers, OpenAI, or similar). Write a
function that takes a query, embeds it, and returns the top-3 most similar chunks using cosine similarity. Test it with 2
sample queries and show the output.


In [20]:
!pip install -q sentence-transformers

import torch
from sentence_transformers import SentenceTransformer

In [39]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus = fixed_overlap_chunks

corpus_embeddings = model.encode_document(
    corpus,
    convert_to_tensor=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [47]:
# Dense retrieval function
def dense_retrieval(query, top_k=3):

    # Create embedding for the query
    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    )

    # Calculate cosine similarity
    similarity_scores = model.similarity(
        query_embedding,
        corpus_embeddings
    )[0]

    # Get top-k most similar chunks
    scores, indices = torch.topk(
        similarity_scores,
        k=min(top_k, len(corpus))
    )

    print("\n" + "=" * 60)
    print("Query:", query)
    print("Top", top_k, "Most Similar Chunks")
    print("=" * 60)

    # Display results
    for rank, (score, index) in enumerate(zip(scores, indices), 1):
        print(f"\nResult {rank}")
        print("Similarity Score:", round(score.item(), 4))
        print("Chunk:")
        print(corpus[index])
        print("-" * 60)

In [44]:
# Test Query 1
dense_retrieval("What is a vector database?", top_k=3)

# Test Query 2
dense_retrieval("Why is chunking important in RAG?", top_k=3)


Query: What is a vector database?
Top 3 most similar chunks:

Score: 0.8512
Chunk: vector database stores data as numerical vectors called embeddings.
It helps applications find information that is similar in meaning instead of only

Score: 0.5959
Chunk: human language.
It is commonly used in chatbots, translation, sentiment analysis, and text summarization.

A vector database stores data as numerical

Score: 0.4339
Chunk: data.
It is used for prediction, classification, recommendation systems, and pattern recognition.

Natural language processing is a field of

Query: Why is chunking important in RAG?
Top 3 most similar chunks:

Score: 0.5397
Chunk: step in a RAG pipeline because large documents are divided into smaller pieces.
Smaller chunks make it easier for the retrieval system to find the

Score: 0.5381
Chunk: it easier for the retrieval system to find the most relevant information.

Chunk overlap means that some part of one chunk is repeated in the next

Score: 0.516
Chunk: som

For this i used this documentation: https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html

B3. Sparse retrieval, BM25 Implement BM25 search over the same chunks (a library like rank_bm25 is fine, no need to
build the scoring from scratch). Run the same 2 queries from B2 through BM25 and compare the results. Which method
worked better for each query, and why do you think that happened?

In [48]:
!pip install rank_bm25

from rank_bm25 import BM25Okapi


In [51]:
corpus = fixed_overlap_chunks # b1 wale chunks liye

tokenized_corpus = [doc.split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus) # model making


In [53]:
query = "What is a vector database?"
tokenized_query = query.split(" ")

# har chunk ka score nikala
doc_scores = bm25.get_scores(tokenized_query)

print("Query 1:", query)
print("BM25 Scores:", doc_scores)


Query 1: What is a vector database?
BM25 Scores: [0.91083551 0.96455213 0.92804332 1.86280015 1.78154715 0.76816259
 0.51095759 0.84809339 0.34477047 0.33785457 0.50332292 0.49754076]


In [54]:
# top 3 chunks nikale
top_documents = bm25.get_top_n(tokenized_query, corpus, n=3)

print("\nTop 3 Results:")
for document in top_documents:
    print("\n", document)


Top 3 Results:

 human language.
It is commonly used in chatbots, translation, sentiment analysis, and text summarization.

A vector database stores data as numerical

 vector database stores data as numerical vectors called embeddings.
It helps applications find information that is similar in meaning instead of only

 data.
It is used for prediction, classification, recommendation systems, and pattern recognition.

Natural language processing is a field of


In [55]:
# dusri query
query = "Why is chunking important in RAG?"
tokenized_query = query.split(" ")

doc_scores = bm25.get_scores(tokenized_query)

print("\n\nQuery 2:", query)
print("BM25 Scores:", doc_scores)

#top 3 chunks nikale
top_documents = bm25.get_top_n(tokenized_query, corpus, n=3)

print("\nTop 3 Results:")
for document in top_documents:
    print("\n", document)



Query 2: Why is chunking important in RAG?
BM25 Scores: [0.53531754 0.56211651 0.92804332 0.76816259 0.73465631 0.76816259
 0.         2.01691253 0.34477047 0.67570914 2.17546497 0.49754076]

Top 3 Results:

 some part of one chunk is repeated in the next chunk.
This helps preserve context when important information is located near the boundary of two

 a knowledge base and provides it to the model as additional context.

Chunking is an important step in a RAG pipeline because large documents are

 language processing is a field of artificial intelligence that helps computers understand human language.
It is commonly used in chatbots,


This code is refer from this link:
https://github.com/dorianbrown/rank_bm25/tree/master